In [ ]:
!pip install langchain langchain-core langchain-openai langchain-community langgraph

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

# -----------------------------
# 1. Documents (your knowledge base)
# -----------------------------
docs = [
    "Python is widely used for AI and machine learning.",
    "LangChain helps build applications using LLMs.",
    "RAG stands for Retrieval-Augmented Generation.",
    "RAG improves accuracy by grounding responses in retrieved data."
]

# -----------------------------
# 2. Split into chunks
# -----------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10
)
documents = splitter.create_documents(docs)

# -----------------------------
# 3. Create embeddings + vector DB
# -----------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)

# -----------------------------
# 4. Create retriever
# -----------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# -----------------------------
# 5. LLM
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# 6. Ask a question
# -----------------------------
query = "What is RAG?"

# Retrieve relevant docs
retrieved_docs = retriever.invoke(query)

# Combine context
context = "\n".join([doc.page_content for doc in retrieved_docs])

# -----------------------------
# 7. Generate answer
# -----------------------------
prompt = f"""
Answer the question using the context below.

Context:
{context}

Question: {query}
"""

response = llm.invoke(prompt)

# -----------------------------
# 8. Output
# -----------------------------
print("Answer:\n", response.content)